**PALEOS MgSiO₃ Equation of State: PROTEUS Lookup Tables**

This notebook generates two pre-computed thermodynamic property tables for MgSiO₃,
designed for the PROTEUS interior model:

1. **Solid table** — regular grid from 300 K to `max(T_liquidus)`, using PALEOS
   phase selection (en, lpcen, hpcen, brg, ppv) at every grid point.
2. **Liquid table** — regular grid from `min(T_solidus)` to 10⁵ K, using Wolf18
   (liquid MgSiO₃) at every grid point.

Both tables share the same pressure grid (0 Pa + log-uniform from 1 bar to 100 TPa)
and use a log-uniform temperature grid at a user-specified resolution (points per decade).
The solidus and liquidus files are used **only** to determine the global T bounds;
every (P, T) point in the rectangular grid is evaluated.

**Author:** Mara Attia  
**Date:** March 2026  

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
import datetime
import os

from paleos import mgsio3_eos as mgsio3

# Pre-initialize one EoS instance per phase to avoid repeated
# construction overhead (Wolf18/RTpress init is expensive).
_eos_instances = {
    'en':     mgsio3.Sokolova22(phase='orthoen'),
    'lpcen':  mgsio3.Sokolova22(phase='lp-cen'),
    'hpcen':  mgsio3.Sokolova22(phase='hp-cen'),
    'brg':    mgsio3.Wolf15(x_Fe=0.0),
    'ppv':    mgsio3.Sakai16(),
    'liquid': mgsio3.Wolf18(),
}

print("PALEOS MgSiO₃ EoS — PROTEUS Table Generator")
print("============================================")

---
# User Input

In [ ]:
# ============================================================================
#  USER INPUT — modify this cell as needed
# ============================================================================

# --- Solidus / liquidus curves (PROTEUS) ---
# Text files with one header line and two columns: P [Pa]  T [K]
# Used ONLY to determine global T bounds for each table.
SOLIDUS_FILE  = "solidus_mantle_proteus.dat"
LIQUIDUS_FILE = "liquidus_mantle_proteus.dat"

# --- Pressure grid ---
P_MIN = 1e5       # Pa  (1 bar, lower end of log grid)
P_MAX = 1e17      # Pa  (100 TPa)
P_ZERO = True     # prepend a single point at P = 0 Pa

# --- Temperature grid ---
T_SOLID_MIN  = 300.0    # K  (lower bound for solid table)
T_LIQUID_MAX = 1e5      # K  (upper bound for liquid table)
# T_SOLID_MAX  is set automatically to max(T_liquidus)
# T_LIQUID_MIN is set automatically to min(T_solidus)

# --- Grid resolution ---
PPD = 150   # points per decade (in both log P and log T)

# --- Output filenames ---
OUTPUT_SOLID  = "mgsio3_tables_proteus_solid.dat"
OUTPUT_LIQUID = "mgsio3_tables_proteus_liquid.dat"

# ============================================================================
print("User input loaded.")

---
# Load Solidus / Liquidus and Determine T Bounds

In [ ]:
def load_boundary(filepath):
    """
    Load a PROTEUS phase-boundary file (one header line, P [Pa]  T [K]).

    Returns P and T arrays sorted by P.
    """
    data = np.loadtxt(filepath, skiprows=1)
    P = data[:, 0]
    T = data[:, 1]
    order = np.argsort(P)
    return P[order], T[order]


# Load
P_sol, T_sol = load_boundary(SOLIDUS_FILE)
P_liq, T_liq = load_boundary(LIQUIDUS_FILE)

# Derive global T bounds
T_SOLID_MAX  = float(np.max(T_liq))   # solid table goes up to max(T_liquidus)
T_LIQUID_MIN = float(np.min(T_sol))   # liquid table starts at min(T_solidus)

print(f"Solidus  file : {SOLIDUS_FILE}  ({len(P_sol)} points)")
print(f"  P range: [{P_sol.min():.2e}, {P_sol.max():.2e}] Pa")
print(f"  T range: [{T_sol.min():.1f}, {T_sol.max():.1f}] K")
print(f"Liquidus file : {LIQUIDUS_FILE}  ({len(P_liq)} points)")
print(f"  P range: [{P_liq.min():.2e}, {P_liq.max():.2e}] Pa")
print(f"  T range: [{T_liq.min():.1f}, {T_liq.max():.1f}] K")
print()
print(f"→ Solid  table T range: [{T_SOLID_MIN:.0f}, {T_SOLID_MAX:.1f}] K")
print(f"→ Liquid table T range: [{T_LIQUID_MIN:.1f}, {T_LIQUID_MAX:.0e}] K")

In [ ]:
# Quick diagnostic plot: solidus, liquidus, and PALEOS melting curve
fig, ax = plt.subplots(figsize=(8, 6), dpi=300)

ax.plot(T_sol, P_sol / 1e9, 'o-', markersize=2, label='PROTEUS solidus', linewidth=1.5)
ax.plot(T_liq, P_liq / 1e9, 's-', markersize=2, label='PROTEUS liquidus', linewidth=1.5)

P_plot = np.logspace(np.log10(P_MIN), np.log10(P_MAX), 500)
ax.plot(np.array([mgsio3.T_melt_MgSiO3(P) for P in P_plot]),
        P_plot / 1e9, '--', label='PALEOS melting (Fei+Belonoshko)',
        linewidth=1.5, alpha=0.7)

# Show derived T bounds as vertical spans
ax.axvline(T_SOLID_MAX, color='C0', linestyle=':', alpha=0.5,
           label=f'T_solid_max = {T_SOLID_MAX:.0f} K')
ax.axvline(T_LIQUID_MIN, color='C1', linestyle=':', alpha=0.5,
           label=f'T_liquid_min = {T_LIQUID_MIN:.0f} K')

ax.set_xlabel('Temperature [K]', fontsize=12)
ax.set_ylabel('Pressure [GPa]', fontsize=12)
ax.set_xscale('log')
ax.set_yscale('log')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, which='both')
ax.set_title('Solidus / Liquidus comparison')
fig.tight_layout()
plt.show()

---
# Build Grids

In [ ]:
def make_log_grid(x_min, x_max, ppd):
    """
    Create a log-uniform grid with specified points per decade.

    Parameters
    ----------
    x_min, x_max : float
        Domain bounds (linear scale, must be > 0).
    ppd : int
        Points per decade.

    Returns
    -------
    log_x : 1-D array
        log10(x) grid values.
    n_points : int
        Total number of points.
    """
    log_min = np.log10(x_min)
    log_max = np.log10(x_max)
    n_decades = log_max - log_min
    n_points = int(np.ceil(n_decades * ppd)) + 1
    log_x = np.linspace(log_min, log_max, n_points)
    return log_x, n_points


# ---- Pressure grid (shared by both tables) ----
log_P_grid, N_P_log = make_log_grid(P_MIN, P_MAX, PPD)
P_grid_log = 10**log_P_grid

if P_ZERO:
    P_grid = np.concatenate(([0.0], P_grid_log))
    N_P = N_P_log + 1
else:
    P_grid = P_grid_log
    N_P = N_P_log

n_decades_P = np.log10(P_MAX) - np.log10(P_MIN)

# ---- Temperature grids (regular rectangular) ----
log_T_solid, N_T_solid = make_log_grid(T_SOLID_MIN, T_SOLID_MAX, PPD)
T_grid_solid = 10**log_T_solid
n_decades_T_solid = np.log10(T_SOLID_MAX) - np.log10(T_SOLID_MIN)

log_T_liquid, N_T_liquid = make_log_grid(T_LIQUID_MIN, T_LIQUID_MAX, PPD)
T_grid_liquid = 10**log_T_liquid
n_decades_T_liquid = np.log10(T_LIQUID_MAX) - np.log10(T_LIQUID_MIN)

# ---- Report ----
print("Grid construction")
print("=" * 60)
print(f"  Resolution: {PPD} points per decade")
print(f"  Pressure:   [{P_grid[0]:.0e}, {P_grid[-1]:.0e}] Pa  "
      f"({n_decades_P:.1f} decades{' + P=0' if P_ZERO else ''})  →  {N_P} points")
print(f"  T (solid):  [{T_SOLID_MIN:.0f}, {T_SOLID_MAX:.1f}] K  "
      f"({n_decades_T_solid:.2f} decades)  →  {N_T_solid} points")
print(f"  T (liquid): [{T_LIQUID_MIN:.1f}, {T_LIQUID_MAX:.0e}] K  "
      f"({n_decades_T_liquid:.2f} decades)  →  {N_T_liquid} points")
print(f"  Solid  grid: {N_P} × {N_T_solid} = {N_P * N_T_solid:,} points")
print(f"  Liquid grid: {N_P} × {N_T_liquid} = {N_P * N_T_liquid:,} points")

---
# Helper Functions

In [ ]:
def phase_to_string(phase):
    """Convert PALEOS phase key to descriptive table string."""
    phase_map = {
        'en':     'solid-en',
        'lpcen':  'solid-lpcen',
        'hpcen':  'solid-hpcen',
        'brg':    'solid-brg',
        'ppv':    'solid-ppv',
        'liquid': 'liquid',
    }
    return phase_map.get(phase, 'unknown')


def get_solid_eos_for_PT(P, T):
    """
    Phase determination among solid phases + pre-initialised lookup.

    Uses the PALEOS phase diagram. If get_mgsio3_phase returns 'liquid'
    (i.e. T exceeds the PALEOS melting curve), we fall back to the solid
    phase just below the melting curve so the solid table is always
    populated with a solid EoS.
    """
    phase = mgsio3.get_mgsio3_phase(P, T)
    if phase == 'liquid':
        T_just_below = mgsio3.T_melt_MgSiO3(P) - 1.0
        if T_just_below < 1.0:
            T_just_below = 1.0
        phase = mgsio3.get_mgsio3_phase(P, T_just_below)
    return _eos_instances[phase], phase


def compute_all_properties(P, T, eos, phase_str):
    """
    Compute all 7 thermodynamic properties for a given EoS at (P, T).

    Returns dict with keys: rho, u, s, cp, cv, alpha, nabla_ad, phase.
    """
    props = {'phase': phase_str}
    for name, method in [('rho', 'density'),
                         ('u', 'specific_internal_energy'),
                         ('s', 'specific_entropy'),
                         ('cp', 'isobaric_heat_capacity'),
                         ('cv', 'isochoric_heat_capacity'),
                         ('alpha', 'thermal_expansion'),
                         ('nabla_ad', 'adiabatic_gradient')]:
        try:
            props[name] = getattr(eos, method)(P, T)
        except Exception:
            props[name] = np.nan
    return props

---
# Generate Solid Table

In [ ]:
print("Generating SOLID lookup table...")
print("=" * 60)

solid_data = []
n_failed_solid = 0
total_solid = N_P * N_T_solid

for i, P in enumerate(P_grid):
    if (i + 1) % max(1, N_P // 20) == 0 or i == 0:
        print(f"  Progress: {i+1}/{N_P} pressure slices "
              f"({100*(i+1)/N_P:.0f}%)")

    for j, T in enumerate(T_grid_solid):
        eos, phase = get_solid_eos_for_PT(P, T)
        props = compute_all_properties(P, T, eos, phase_to_string(phase))

        n_nan = sum(1 for k in ['rho', 'u', 's', 'cp', 'cv', 'alpha', 'nabla_ad']
                    if np.isnan(props[k]))
        if n_nan > 0:
            n_failed_solid += 1
        else:
            solid_data.append({'P': P, 'T': T, **props})

print(f"\nSolid table complete:")
print(f"  Grid points:  {total_solid:,}")
print(f"  Failed (NaN): {n_failed_solid:,} ({100*n_failed_solid/total_solid:.2f}%)")
print(f"  Written rows: {len(solid_data):,}")

---
# Generate Liquid Table

In [ ]:
print("Generating LIQUID lookup table...")
print("=" * 60)

liquid_data = []
n_failed_liquid = 0
total_liquid = N_P * N_T_liquid
eos_liquid = _eos_instances['liquid']

for i, P in enumerate(P_grid):
    if (i + 1) % max(1, N_P // 20) == 0 or i == 0:
        print(f"  Progress: {i+1}/{N_P} pressure slices "
              f"({100*(i+1)/N_P:.0f}%)")

    for j, T in enumerate(T_grid_liquid):
        props = compute_all_properties(P, T, eos_liquid, 'liquid')

        n_nan = sum(1 for k in ['rho', 'u', 's', 'cp', 'cv', 'alpha', 'nabla_ad']
                    if np.isnan(props[k]))
        if n_nan > 0:
            n_failed_liquid += 1
        else:
            liquid_data.append({'P': P, 'T': T, **props})

print(f"\nLiquid table complete:")
print(f"  Grid points:  {total_liquid:,}")
print(f"  Failed (NaN): {n_failed_liquid:,} ({100*n_failed_liquid/total_liquid:.2f}%)")
print(f"  Written rows: {len(liquid_data):,}")

---
# Write Tables to File

In [ ]:
def generate_header(table_type, T_min, T_max, N_T, n_rows, n_failed):
    """
    Generate a comprehensive header for a PROTEUS lookup table.

    Parameters
    ----------
    table_type : str
        'solid' or 'liquid'
    T_min, T_max : float
        Temperature grid bounds [K]
    N_T : int
        Number of temperature grid points
    n_rows : int
        Number of valid rows written
    n_failed : int
        Number of points that failed (NaN)
    """
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S UTC")

    if table_type == 'solid':
        phase_block = (
            "# Phase Codes:\n"
            "#   solid-en    : Orthoenstatite (Pbca)\n"
            "#   solid-lpcen : Low-pressure clinoenstatite (P2\u2081/c)\n"
            "#   solid-hpcen : High-pressure clinoenstatite (C2/c)\n"
            "#   solid-brg   : Bridgmanite (Pbnm)\n"
            "#   solid-ppv   : Post-perovskite (Cmcm)\n"
        )
        eos_block = (
            "# EoS Sources:\n"
            "#   En, LP-CEn, HP-CEn : Sokolova et al. (2022)\n"
            "#   Bridgmanite        : Wolf et al. (2015)\n"
            "#   Post-perovskite    : Sakai et al. (2016)\n"
            "#   Phase boundaries   : Sokolova et al. (2022), Ono & Oganov (2005)\n"
        )
        t_bound_note = (
            f"#   T_max = {T_max:.2f} K = max(T_liquidus) from {LIQUIDUS_FILE}\n"
        )
    else:
        phase_block = (
            "# Phase Codes:\n"
            "#   liquid : Liquid MgSiO\u2083\n"
        )
        eos_block = (
            "# EoS Sources:\n"
            "#   Liquid : Wolf & Bower (2018) [Luo & Deng 2025 params]\n"
        )
        t_bound_note = (
            f"#   T_min = {T_min:.2f} K = min(T_solidus) from {SOLIDUS_FILE}\n"
        )

    P0_note = ""
    if P_ZERO:
        P0_note = (
            "#   The first pressure slice is P = 0 Pa (outside the log grid).\n"
        )

    header = (
        f"# ==============================================================================\n"
        f"# PALEOS MgSiO\u2083 {table_type.capitalize()} EoS Lookup Table \u2014 PROTEUS\n"
        f"# ==============================================================================\n"
        f"#\n"
        f"# Description:\n"
        f"#   Pre-computed thermodynamic properties for {table_type} MgSiO\u2083\n"
        f"#   on a regular (P, T) grid for the PROTEUS interior model.\n"
        f"#\n"
        f"# Generation Info:\n"
        f"#   Generated: {timestamp}\n"
        f"#   Resolution: {PPD} points per decade (log-uniform in P and T)\n"
        f"#   P grid: {N_P} points  [{P_grid[0]:.8e}, {P_grid[-1]:.8e}] Pa\n"
        f"#   T grid: {N_T} points  [{T_min:.8e}, {T_max:.8e}] K\n"
        f"#   Grid size: {N_P} x {N_T} = {N_P * N_T:,} points\n"
        f"#              ({n_rows:,} valid / {n_failed:,} skipped due to non-convergence)\n"
        f"#\n"
        f"# Temperature Bound:\n"
        f"{t_bound_note}"
        f"#\n"
        f"{P0_note}"
        f"#\n"
        f"# Interpolation Notes:\n"
        f"#   - Perform bilinear interpolation in (log10(P), log10(T)) space\n"
        f"#     (use a separate bin for P = 0 if present)\n"
        f"#   - Points where the EoS failed to converge are omitted from the table;\n"
        f"#     the grid is therefore not strictly regular. When reconstructing the\n"
        f"#     full grid, fill missing entries with NaN.\n"
        f"#\n"
        f"{phase_block}"
        f"#\n"
        f"{eos_block}"
        f"#\n"
        f"# Columns:\n"
        f"#   1. P         - Pressure [Pa]\n"
        f"#   2. T         - Temperature [K]\n"
        f"#   3. rho       - Density [kg/m\u00b3]\n"
        f"#   4. u         - Specific internal energy [J/kg]\n"
        f"#   5. s         - Specific entropy [J/(kg\u00b7K)]\n"
        f"#   6. cp        - Isobaric heat capacity [J/(kg\u00b7K)]\n"
        f"#   7. cv        - Isochoric heat capacity [J/(kg\u00b7K)]\n"
        f"#   8. alpha     - Thermal expansion coefficient [K\u207b\u00b9]\n"
        f"#   9. nabla_ad  - Adiabatic gradient (d ln T / d ln P)_S [dimensionless]\n"
        f"#  10. phase     - Phase identifier (see Phase Codes above)\n"
        f"#\n"
        f"# Missing Values:\n"
        f"#   Rows where the EoS failed to converge are omitted entirely.\n"
        f"#   When reconstructing the full grid, fill missing entries with NaN.\n"
        f"#\n"
        f"# ==============================================================================\n"
    )
    return header


def write_table(filename, data, header):
    """Write a lookup table with header, column names, and data rows."""
    with open(filename, 'w') as f:
        f.write(header)
        f.write("# P T rho u s cp cv alpha nabla_ad phase\n")
        for row in data:
            def fmt(x):
                return "NaN".rjust(16) if np.isnan(x) else f"{x:.8e}"
            line = (f"{fmt(row['P'])} {fmt(row['T'])} {fmt(row['rho'])} "
                    f"{fmt(row['u'])} {fmt(row['s'])} {fmt(row['cp'])} "
                    f"{fmt(row['cv'])} {fmt(row['alpha'])} {fmt(row['nabla_ad'])} "
                    f"{row['phase']}\n")
            f.write(line)
    size_mb = os.path.getsize(filename) / 1e6
    print(f"  Written: {filename}  ({size_mb:.1f} MB, {len(data):,} rows)")

In [ ]:
print("Writing tables...")
print("=" * 60)

# Solid
header_solid = generate_header(
    table_type='solid',
    T_min=T_SOLID_MIN, T_max=T_SOLID_MAX, N_T=N_T_solid,
    n_rows=len(solid_data), n_failed=n_failed_solid,
)
write_table(OUTPUT_SOLID, solid_data, header_solid)

# Liquid
header_liquid = generate_header(
    table_type='liquid',
    T_min=T_LIQUID_MIN, T_max=T_LIQUID_MAX, N_T=N_T_liquid,
    n_rows=len(liquid_data), n_failed=n_failed_liquid,
)
write_table(OUTPUT_LIQUID, liquid_data, header_liquid)

print("\nDone!")

---
# Verification

In [ ]:
def verify_table(filename, label):
    """Read back a table and print summary statistics."""
    print(f"\nVerification: {label} ({filename})")
    print("=" * 60)

    numeric_cols = ['P', 'T', 'rho', 'u', 's', 'cp', 'cv', 'alpha', 'nabla_ad']
    data = np.genfromtxt(filename, names=numeric_cols + ['phase'],
                         dtype=None, encoding='utf-8')

    print(f"Rows: {len(data):,}")

    # NaN counts
    print("\nNaN counts:")
    for col in numeric_cols:
        n_nan = np.sum(np.isnan(data[col].astype(float)))
        print(f"  {col:10s}: {n_nan:6d} ({100*n_nan/len(data):.2f}%)")

    # Phase distribution
    phases, counts = np.unique(data['phase'], return_counts=True)
    print("\nPhase distribution:")
    for phase, count in zip(phases, counts):
        print(f"  {phase:20s}: {count:6d} ({100*count/len(data):.1f}%)")

    # Value ranges
    print("\nValue ranges (excluding NaN):")
    for col in ['rho', 'u', 's', 'cp', 'cv', 'alpha', 'nabla_ad']:
        vals = data[col].astype(float)
        valid = vals[~np.isnan(vals)]
        if len(valid) > 0:
            print(f"  {col:10s}: [{valid.min():.3e}, {valid.max():.3e}]")


verify_table(OUTPUT_SOLID,  "Solid table")
verify_table(OUTPUT_LIQUID, "Liquid table")

In [ ]:
# Diagnostic plot: density across both tables
fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=300)

for ax, data_list, title in [(axes[0], solid_data, 'Solid'),
                              (axes[1], liquid_data, 'Liquid')]:
    P_arr = np.array([r['P'] for r in data_list])
    T_arr = np.array([r['T'] for r in data_list])
    rho_arr = np.array([r['rho'] for r in data_list])

    # Replace P=0 with a small value for log plotting
    #P_plot = np.where(P_arr > 0, P_arr, 1e3)
    P_plot = P_arr

    sc = ax.scatter(T_arr, P_plot / 1e9, c=np.log10(rho_arr),
                    s=0.2, cmap='viridis', rasterized=True)
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label('$log_{10}$($\\rho$ [kg/m$^3$])', fontsize=10)

    # Overlay solidus / liquidus from file
    ax.plot(T_sol, P_sol / 1e9, 'r-', linewidth=1.5, label='solidus')
    ax.plot(T_liq, P_liq / 1e9, 'r--', linewidth=1.5, label='liquidus')

    ax.loglog()
    ax.set_xlim(300, 1e5)
    ax.set_ylim(1e-4, 1e5)
    ax.set_xlabel('Temperature [K]', fontsize=12)
    ax.set_ylabel('Pressure [GPa]', fontsize=12)
    ax.set_title(f'{title} table — density', fontsize=13)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, which='both')

fig.tight_layout()
plt.show()